# MSc. Dissertation - DataScience By Robert Solomon
## BACKGROUND:   
### Analysis of Remote Work Impact on Employee Well-Being (Secondary Dataset)
This notebook focuses on cleaning, processing, and preparing the secondary dataset obtained from Kaggle that explores the effects of remote work on employee mental health, work-life balance, and productivity etc. as part of MSc. Dissertation. The dataset includes various factors such as work location (remote/hybrid/on-site), hours worked per week, stress levels, mental health conditions, and access to employer-provided support resources.

To ensure the data is structured for statistical analysis, the following data preparation steps were performed:

- Handling missing values by imputing categorical and numerical data appropriately.
- Encoding categorical variables (e.g., work location, mental health condition, and gender) using label encoding and one-hot encoding.
- Standardizing numerical values for consistency across features such as work-life balance ratings and stress levels.
- Detecting outliers for predictive modelling, however table wasn't removed because it caused a massive data loss.
- Feature engineering to create new insights, such as a Work Stress Score, to enhance statistical analysis.
- This cleaned dataset will later be used for correlation analysis, descriptive analysis, anova tests ,regression modeling, and comparative studies with primary survey data collected from remote workers. The insights derived from this analysis will contribute to a deeper understanding of how remote work impacts employee satisfaction, social isolation, and overall productivity.

In [ ]:
# Importing necessary libraries here below:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Loading in the secondary data (Kaggle)

In [ ]:
wfh_mentalHealth_data = pd.read_csv('../Secondary_Research/SR_Dataset/Remote_Work_on_Mental_Health.csv')

In [ ]:
# Display the first few rows to inspect the data
wfh_mentalHealth_data

In [ ]:
# Checking number of rows, columns in the data.

print(wfh_mentalHealth_data.shape)

In [ ]:
# Describing (summarize) all variables.

wfh_mentalHealth_data.describe()

## 1.1 Checking the data types and missing values:

In [ ]:
# Getting dataset info
wfh_mentalHealth_data.info()

In [ ]:
# Checking for missing values in each column
wfh_mentalHealth_data.isnull().sum()

## 2. Handling Missing Data

### The dataset after observation and partial investigation indicates some missing values, particularly in:

- Mental_Health_Condition
- Physical_Activity

We can handle missing values as follows:

### 2.1 Drop Rows with Too Many Missing Values
Given that some rows have excessive missing values (e.g., 3 or more NaNs), they might not be useful for my analysis so I will drop them.

In [ ]:
# Dropping rows where there's missing values in 'Mental_Health_Condition' column and 'Physical_Activity' column:
wfh_mentalHealth_data.dropna(subset=['Mental_Health_Condition'], inplace=True)
wfh_mentalHealth_data.dropna(subset=['Physical_Activity'], inplace=True)  

In [ ]:
# Re-checking again for missing values after dropping rows
wfh_mentalHealth_data.isnull().sum()

In [ ]:
# Re-checking number of rows, columns in the data post dropping of rows.
print(wfh_mentalHealth_data.shape)

In [ ]:
# Display the first few rows to inspect the data again for reassurance
wfh_mentalHealth_data.head()

# 3. Converting Categorical Variables
To ensure statistical compatibility, we must encode categorical data.

### 3.1 Gender (Categorical)
- This column contains "Male", "Female".
- Since gender is categorical but not ordinal, I will use Label Encoding for simplicity and focus on gender that makes sense, 'Male' and 'Female'.

In [ ]:
wfh_mentalHealth_data['Gender'] = wfh_mentalHealth_data['Gender'].map({'Male': 0, 'Female': 1})

In [ ]:
# Checking again for missing values after dropping rows
wfh_mentalHealth_data.isnull().sum()

### 3.2 Mental_Health_Condition (Categorical)
- This column contains "Anxiety", "Depression", "None", etc.
- Since this is nominal (not ordered), we can use One-Hot Encoding.



One-Hot Encoding Mental Health Condition:

In [ ]:
wfh_mentalHealth_data = pd.get_dummies(wfh_mentalHealth_data, columns=['Mental_Health_Condition'], prefix='MHC', drop_first=True)

### 3.3 Converting Binary Categories to 0/1
- For Yes/No columns like "Access_to_Mental_Health_Resources", I'm converting them to 0s and 1s.

In [ ]:
wfh_mentalHealth_data['Access_to_Mental_Health_Resources'] = wfh_mentalHealth_data['Access_to_Mental_Health_Resources'].map({'Yes': 1, 'No': 0})

In [ ]:
wfh_mentalHealth_data.head()

### 3.4 Encoding Multi-Class Categories

For categorical columns like Job_Role, Industry, and Region, we use One-Hot Encoding.

In [ ]:
wfh_mentalHealth_data = pd.get_dummies(wfh_mentalHealth_data, columns=['Job_Role', 'Industry', 'Region'], drop_first=True)

For ordinal categories like Work_Location, we map them to numerical values:

In [ ]:
wfh_mentalHealth_data['Work_Location'] = wfh_mentalHealth_data['Work_Location'].map({'Remote': 1, 'Hybrid': 2, 'Onsite': 3})

For Stress Level and Satisfaction with Remote Work, we assign numerical rankings:

In [ ]:
wfh_mentalHealth_data['Stress_Level'] = wfh_mentalHealth_data['Stress_Level'].map({'Low': 1, 'Medium': 2, 'High': 3})
wfh_mentalHealth_data['Satisfaction_with_Remote_Work'] = wfh_mentalHealth_data['Satisfaction_with_Remote_Work'].map({'Unsatisfied': 1, 'Neutral': 2, 'Satisfied': 3})

In [ ]:
# Validating number of columns ,post encoding above:
print(wfh_mentalHealth_data.shape)

In [ ]:
wfh_mentalHealth_data.head()

In [ ]:
# Dropping rows where there's missing values, in this case, 'Gender' column:
wfh_mentalHealth_data.dropna(subset=['Gender'], inplace=True)

In [ ]:
# Re-checking again for missing values
wfh_mentalHealth_data.isnull().sum()

Writing to excel file after all the encoding was carried out for further validation:

In [ ]:
wfh_mentalHealth_data.to_csv("../Secondary_Research/SR_Dataset/post_Encoded_Remote_Work_on_Mental_Health.csv", index=False)

# 4. Standardizing Numerical Data
Here we are ensuring that numerical columns are in the proper format.

In [ ]:
# Converting necessary columns to numeric
numeric_cols = ['Age', 'Years_of_Experience', 'Hours_Worked_Per_Week', 
                'Work_Life_Balance_Rating', 'Stress_Level', 'Social_Isolation_Rating']


# For loop to iterate through the 'numeric_cols' array object of my chosen columns subject to proper formatting: 
for col in numeric_cols:
    wfh_mentalHealth_data[col] = pd.to_numeric(wfh_mentalHealth_data[col], errors='coerce')

In [ ]:
wfh_mentalHealth_data

In [ ]:
# Re-checking again for missing values
wfh_mentalHealth_data.isnull().sum()

In [ ]:
wfh_mentalHealth_data

# 5. Handling Outliers
Checking for outliers using the Interquartile Range (IQR) Method.

In [ ]:
# Defining IQR range
Q1 = wfh_mentalHealth_data[numeric_cols].quantile(0.25)
Q3 = wfh_mentalHealth_data[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

#Printing output:
# IQR

Q1

In [ ]:
Q3

In [ ]:
IQR

In [ ]:
# Re-checking again for missing values
# wfh_mentalHealth_data.isnull().sum()

In [ ]:
# Defining any outlier boundaries
lower_bound = Q1 - 2.5 * IQR
upper_bound = Q3 + 2.5 * IQR

In [ ]:
# Printing a boxplot showing the outliers
wfh_mentalHealth_data.boxplot(rot=90, figsize=(10,5))
plt.show()

# 6. Feature Engineering
This is not particularly necessary but I've applied feature engineering to predict outcomes based on certain factors (variables) in the dataset.

In [ ]:
# Creating a 'Work_Stress_Score' based on multiple factors (Hours worked per week, number of virtual meetings and work life balance rating) to possibly enhance statistical analysis.
wfh_mentalHealth_data['Work_Stress_Score'] = wfh_mentalHealth_data['Hours_Worked_Per_Week'] + wfh_mentalHealth_data['Number_of_Virtual_Meetings'] - wfh_mentalHealth_data['Work_Life_Balance_Rating']

In [ ]:
# Displaying snippet of cleaned dataset for reassurance: 
wfh_mentalHealth_data

In [ ]:
# Re-checking again for missing values
wfh_mentalHealth_data.isnull().sum()

# 7.  Saving Cleaned Dataset
Saving pre-processed dataset, which is ready to be used for analysis:

In [ ]:
wfh_mentalHealth_data.to_csv("../Secondary_Research/SR_Dataset/cleaned_Remote_Work_on_Mental_Health.csv", index=False)